# DPE: compare several weighting rules from one saved run

Run this notebook top to bottom to fit power, softmax, top-k and regularized joint weights on the **same cached head probabilities**. No backbone, feature extraction or prototype training is repeated. CPU execution is sufficient.

First complete `dpe_head_reweighting.ipynb` once, or extract its results ZIP. Keep that source run intact. This notebook saves a separate suite. B fits weights; C selects one configuration per family and an overall configuration, each with an exact uniform fallback. All decisions are saved before test predictions are opened. Reusing seed 0 after its test results were inspected is exploratory.

See [method instructions](../src/methods/dpe_head_reweighting/README.md#multiple-weighting-rules-without-retraining). The source bundle must include its manifest, original selection, ensemble checkpoint and B/C/test prediction caches. Raw images, backbone and raw features are unnecessary here.


In [ ]:
import importlib.util
import os
from pathlib import Path
import subprocess
import sys

REPO_PATH = None  # Set to your repository checkout if automatic discovery fails.
working = Path.cwd().resolve()
candidates = [working, *working.parents, working / "waterbirds-shortcut-mitigation"]
REPO = Path(REPO_PATH).expanduser().resolve() if REPO_PATH else next(
    (p for p in candidates if (p / "scripts/dpe_weighting_rules.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Set REPO_PATH to the checkout containing scripts/dpe_weighting_rules.py.")
missing = [package for module, package in [
    ("numpy", "numpy>=1.26,<3"), ("scipy", "scipy>=1.11,<2"),
    ("torch", "torch>=2.6,<3"), ("yaml", "PyYAML>=6,<7"), ("pandas", "pandas>=2.2,<3")]
    if importlib.util.find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
import json
import pandas as pd
from IPython.display import display
from src.methods.dpe_head_reweighting.rule_suite import select, evaluate
print("Repository:", REPO)


## 1. Source run and fixed comparison settings

Point `SOURCE_RUN_DIR` at the extracted original run (the directory containing `experiment/`, `checkpoints/`, and `results/`). Give each independent comparison a new `SUITE_NAME`. The source is read only; output must be outside that run.

The YAML predeclares seven new candidates: softmax beta 10/30, top-k 10/5, and joint lambda 1/0.1/0.01. Uniform and all 12 original power candidates remain controls. Declare any grid changes **before test evaluation**, using a fresh suite directory. Top-k score ties use source head order. Complete C accuracy ties prefer uniform, then declaration order.


In [ ]:
SOURCE_RUN_DIR = (REPO / "outputs/dpe_head_reweighting_runs/h2-standard-seed0").resolve()
SUITE_NAME = "h2-standard-seed0-multiple-rules-v1"
CONFIG_PATH = REPO / "configs/dpe_head_reweighting/weighting_rules.yaml"
OUTPUT_DIR = (REPO / "outputs/dpe_weighting_rule_runs" / SUITE_NAME).resolve()
if not SOURCE_RUN_DIR.is_dir():
    raise FileNotFoundError("Extract your H2 results ZIP and set SOURCE_RUN_DIR, or finish the original H2 notebook first.")
required = ["experiment/manifest.json", "experiment/selection.json",
            "checkpoints/ensemble.pt", "experiment/predictions/B.npz",
            "experiment/predictions/C.npz", "experiment/predictions/test.npz"]
missing = [name for name in required if not (SOURCE_RUN_DIR / name).is_file()]
if missing:
    raise FileNotFoundError("SOURCE_RUN_DIR must point to the extracted run containing experiment/ and checkpoints/. Missing: " + ", ".join(missing))
print("Read-only source:", SOURCE_RUN_DIR)
print("Separate output:", OUTPUT_DIR)
print(CONFIG_PATH.read_text())


## 2. Fit on B and freeze selection on C

All rules reuse one set of B/C probabilities. Joint weighting minimizes maximum group mean log loss on B plus squared distance from uniform, with nonnegative weights summing to one. A failed optimizer stops the suite rather than silently dropping a candidate.

This stage does not require or open test predictions. Re-running checks saved selection against source hashes, code, settings and recomputed B/C fits. It rejects changed inputs instead of overwriting a completed experiment.


In [ ]:
selection = select(SOURCE_RUN_DIR, CONFIG_PATH, OUTPUT_DIR)
print("Source track/seed:", selection["source_track"], selection["source_seed"])
print("C-selected overall configuration:", selection["selected"])
print("C-selected configuration by family:", selection["per_family_selected"])
comparison = pd.DataFrame([{
    "candidate": row["id"], "rule": row["family"],
    "C WGA": row["metrics"]["wga"], "C accuracy": row["metrics"]["overall"],
    "effective heads": row["effective_heads"],
    "changed C predictions": row["c_prediction_changes"],
    "maximum C probability change": row["c_probability_max_abs_delta"],
} for row in selection["candidates"]])
display(comparison)
weights = pd.DataFrame({row["id"]: row["weights"] for row in selection["candidates"]},
                       index=selection["head_ids"])
display(weights)


## 3. Evaluate the frozen family winners

Only the uniform control, C-selected winner for each family, and C-selected overall winner are evaluated. The table is a comparison, not permission to choose a method using test WGA. If a family retains uniform, its identical results are valid.

The source test cache must share the ensemble identity and have no validation ID overlap. Selection and test-cache hashes bind saved results. Keep already inspected seed-0 findings exploratory and freeze the protocol before later seeds.


In [ ]:
result = evaluate(SOURCE_RUN_DIR, CONFIG_PATH, OUTPUT_DIR)
test_table = pd.DataFrame([{
    "rule": family, "C-selected candidate": row["candidate"],
    "test accuracy": row["metrics"]["overall"], "test WGA": row["metrics"]["wga"],
    "WGA change (pp)": row["delta_pp"]["wga"],
    "accuracy change (pp)": row["delta_pp"]["overall"],
    "changed test predictions": row["changed_predictions"],
    "effective heads": row["effective_heads"],
} for family, row in result["results"].items()])
display(test_table)
group_table = pd.DataFrame([{
    "rule": family, "group": group, "accuracy": accuracy,
    "count": row["metrics"]["counts"][group],
} for family, row in result["results"].items() for group, accuracy in row["metrics"]["groups"].items()])
display(group_table)
print("Report:", OUTPUT_DIR / "report.md")


## 4. Export this comparison

The new ZIP contains candidate metrics, exact fitted weights, solver diagnostics, selected test results, per-group counts, paired differences, the resolved rule configuration, source hashes and code/runtime identities. It does not duplicate the original heads or prediction caches: retain the source H2 ZIP separately.

Repeat with the corresponding original source run for seeds 1 and 2 and a new suite name per seed. No real Waterbirds result for these new rules is claimed until those runs are executed and documented.


In [ ]:
import shutil
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Suite archive:", archive)
print("Retain the original H2 source archive separately.")
